# Transfer Learning Feasibility Classification — TrAdaBoost / KMM / Optimal Transport

Three instance/feature-based domain adaptation methods, one per source task (plus a multi-source OT method):
1. **TrAdaBoost** (ADAPT) — supervised reweighting using target labels
2. **KMM** (ADAPT) — unsupervised feature-distribution matching (RBF kernel)
3. **SinkhornL1l2Transport** (POT) — per-source class-regularized optimal transport
4. **JCPOTTransport** (POT) — multi-source OT (Redko et al. 2019, the JDOT-family multi-source extension)

- Base estimator: `DecisionTreeClassifier`
- Target task: `150_ST150MD0-N2H0-30-2_L76_0.4_10000_H450`
- All compared against a **no-transfer baseline** (target-only classifier)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from adapt.instance_based import TrAdaBoost, KMM

SEED = 42
np.random.seed(SEED)

## 1. Load data and select target / source tasks

In [ ]:
with open("./data/task_df_dict.pkl", "rb") as f:
    task_df_dict = pickle.load(f)

FEATURE_COLS = ["feedrate", "gas_pressure", "focal_position"]
TARGET_TASK = "150_ST150MD0-N2H0-30-2_L76_0.4_10000_H450"

def make_feasibility_label(df):
    return ((df["burr_evaluated"] >= 0) & (df["roughness_z_evaluated"] >= 0)).astype(int).values

# Source tasks: must have both classes (feasible & infeasible) and >= 5 points
source_tasks = []
for name, df in task_df_dict.items():
    if name == TARGET_TASK:
        continue
    y = make_feasibility_label(df)
    if len(np.unique(y)) == 2 and len(df) >= 5:
        source_tasks.append(name)

print(f"Target task: {TARGET_TASK} ({len(task_df_dict[TARGET_TASK])} points)")
print(f"Source tasks: {len(source_tasks)}")

## 2. Build target train/test split (small target sample to simulate transfer scenario)

In [ ]:
df_target = task_df_dict[TARGET_TASK]
X_target_all = df_target[FEATURE_COLS].values
y_target_all = make_feasibility_label(df_target)

# Simulate the BO scenario: only a few labeled target samples
N_TARGET_LABELED = 10

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target_all, y_target_all,
    train_size=N_TARGET_LABELED,
    stratify=y_target_all,
    random_state=SEED,
)

# Standardize using target training data
scaler = StandardScaler().fit(X_target_train)
X_target_train_s = scaler.transform(X_target_train)
X_target_test_s = scaler.transform(X_target_test)

print(f"Target train: {len(y_target_train)} ({y_target_train.sum()} feasible)")
print(f"Target test:  {len(y_target_test)} ({y_target_test.sum()} feasible)")

## 3. Prepare source-task data (each source kept separate)

In [ ]:
source_data = {}
for name in source_tasks:
    df = task_df_dict[name]
    X = scaler.transform(df[FEATURE_COLS].values)  # use target scaler so domains align
    y = make_feasibility_label(df)
    source_data[name] = (X, y)
    print(f"  {name}: {len(y)} pts, feasible_rate={y.mean():.2f}")

## 4. Train one TrAdaBoost classifier per source task

Each model is trained with:
- `X, y` = source task data
- `Xt, yt` = target task labeled data (small)

The base estimator is a shallow decision tree.

In [ ]:
def make_base_estimator():
    return DecisionTreeClassifier(max_depth=5, random_state=SEED)

models = {}
for name, (Xs, ys) in source_data.items():
    try:
        model = TrAdaBoost(
            estimator=make_base_estimator(),
            n_estimators=20,
            Xt=X_target_train_s,
            yt=y_target_train,
            random_state=SEED,
            verbose=0,
        )
        model.fit(Xs, ys)
        models[name] = model
    except Exception as e:
        print(f"  Failed for {name}: {e}")

print(f"\nTrained {len(models)} TrAdaBoost models.")

## 5. Per-source evaluation on target test set

In [ ]:
per_source_results = []
for name, model in models.items():
    y_pred = model.predict(X_target_test_s)
    y_pred = np.asarray(y_pred).astype(int).ravel()
    acc = accuracy_score(y_target_test, y_pred)
    f1 = f1_score(y_target_test, y_pred, zero_division=0)
    per_source_results.append({"source": name, "accuracy": acc, "f1": f1})

results_df = pd.DataFrame(per_source_results).sort_values("accuracy", ascending=False)
print(results_df.to_string(index=False))

## 6. Ensemble: average predicted probability across all source models

In [ ]:
def predict_proba_safe(model, X):
    """TrAdaBoost may not expose predict_proba on the base; fall back to hard labels."""
    if hasattr(model, "predict_proba"):
        try:
            p = model.predict_proba(X)
            p = np.asarray(p)
            if p.ndim == 2 and p.shape[1] == 2:
                return p[:, 1]
            return p.ravel()
        except Exception:
            pass
    return model.predict(X).astype(float).ravel()

probs = np.stack([predict_proba_safe(m, X_target_test_s) for m in models.values()], axis=0)
ensemble_proba = probs.mean(axis=0)
ensemble_pred = (ensemble_proba >= 0.5).astype(int)

ens_acc = accuracy_score(y_target_test, ensemble_pred)
ens_f1 = f1_score(y_target_test, ensemble_pred, zero_division=0)
try:
    ens_auc = roc_auc_score(y_target_test, ensemble_proba)
except ValueError:
    ens_auc = float("nan")

print(f"Ensemble accuracy: {ens_acc:.3f}")
print(f"Ensemble F1:       {ens_f1:.3f}")
print(f"Ensemble AUC:      {ens_auc:.3f}")

## 7. No-transfer baseline (target-only)

In [ ]:
baseline = make_base_estimator()
baseline.fit(X_target_train_s, y_target_train)
baseline_pred = baseline.predict(X_target_test_s)
baseline_proba = baseline.predict_proba(X_target_test_s)[:, 1]

base_acc = accuracy_score(y_target_test, baseline_pred)
base_f1 = f1_score(y_target_test, baseline_pred, zero_division=0)
try:
    base_auc = roc_auc_score(y_target_test, baseline_proba)
except ValueError:
    base_auc = float("nan")

print(f"Baseline (no transfer) accuracy: {base_acc:.3f}")
print(f"Baseline F1:                     {base_f1:.3f}")
print(f"Baseline AUC:                    {base_auc:.3f}")

print("\n--- Comparison ---")
print(f"{'Method':<25} {'Acc':>6} {'F1':>6} {'AUC':>6}")
print(f"{'No-transfer baseline':<25} {base_acc:>6.3f} {base_f1:>6.3f} {base_auc:>6.3f}")
print(f"{'TrAdaBoost ensemble':<25} {ens_acc:>6.3f} {ens_f1:>6.3f} {ens_auc:>6.3f}")

## 8. Visualize per-source accuracy

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=results_df["source"], y=results_df["accuracy"], name="TrAdaBoost (per source)",
    marker_color="steelblue",
))
fig.add_hline(y=base_acc, line_dash="dash", line_color="red",
              annotation_text=f"baseline = {base_acc:.3f}", annotation_position="top right")
fig.add_hline(y=ens_acc, line_dash="dash", line_color="green",
              annotation_text=f"ensemble = {ens_acc:.3f}", annotation_position="bottom right")
fig.update_layout(
    title="TrAdaBoost Target-Test Accuracy per Source Task",
    xaxis_title="Source task", yaxis_title="Accuracy",
    xaxis_tickangle=-45, height=500, width=1100,
    yaxis=dict(range=[0, 1]),
)
fig.show()

## 9. Visualize ensemble decision boundary (focal_position fixed)

In [ ]:
# --- Build a 3D grid over the full target feature box ---
n_grid = 25  # per-axis resolution (n_grid**3 evaluation points)
fr = np.linspace(df_target["feedrate"].min(),       df_target["feedrate"].max(),       n_grid)
gp = np.linspace(df_target["gas_pressure"].min(),   df_target["gas_pressure"].max(),   n_grid)
fp = np.linspace(df_target["focal_position"].min(), df_target["focal_position"].max(), n_grid)

FR3, GP3, FP3 = np.meshgrid(fr, gp, fp, indexing="ij")
grid3 = np.column_stack([FR3.ravel(), GP3.ravel(), FP3.ravel()])
grid_s = scaler.transform(grid3)

# Keep a 2D slice grid for any remaining 2D plots (and for legacy variables `gp`, `fr`)
fp_fixed = float(df_target["focal_position"].median())


def plot_decision_isosurface(p_grid_flat, title):
    """Render a 3D iso-surface of P(feasible) = 0.5, colored by P, with target points overlaid."""
    fig = go.Figure()

    # Iso-surface (decision boundary) + colored volume of P(feasible)
    fig.add_trace(go.Isosurface(
        x=FR3.ravel(), y=GP3.ravel(), z=FP3.ravel(),
        value=p_grid_flat,
        isomin=0.3, isomax=0.7,
        surface_count=3,
        colorscale="RdYlGn", cmin=0, cmax=1,
        opacity=0.5,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorbar=dict(title="P(feasible)"),
    ))

    # Target training points
    fig.add_trace(go.Scatter3d(
        x=X_target_train[feas_tr, 0], y=X_target_train[feas_tr, 1], z=X_target_train[feas_tr, 2],
        mode="markers",
        marker=dict(size=5, color="green", symbol="circle", line=dict(width=1, color="black")),
        name="Target train (feasible)",
    ))
    fig.add_trace(go.Scatter3d(
        x=X_target_train[~feas_tr, 0], y=X_target_train[~feas_tr, 1], z=X_target_train[~feas_tr, 2],
        mode="markers",
        marker=dict(size=5, color="red", symbol="x"),
        name="Target train (infeasible)",
    ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="Feedrate",
            yaxis_title="Gas Pressure",
            zaxis_title="Focal Position",
        ),
        width=900, height=700,
    )
    return fig


# --- TrAdaBoost ensemble in 3D ---
feas_tr = y_target_train == 1
probs_grid = np.stack([predict_proba_safe(m, grid_s) for m in models.values()], axis=0)
p_tradaboost = probs_grid.mean(axis=0)

plot_decision_isosurface(
    p_tradaboost,
    f"TrAdaBoost Ensemble — P(feasible) iso-surfaces"
).show()

## 10. Train one KMM classifier per source task

KMM reweights source samples so that their **distribution** matches the target in an RBF kernel feature space — it does **not** use target labels. The base estimator is then fit on the reweighted source data.

In [ ]:
kmm_models = {}
for name, (Xs, ys) in source_data.items():
    try:
        model = KMM(
            estimator=make_base_estimator(),
            Xt=X_target_train_s,
            kernel="rbf",
            B=1000,
            max_size=1000,
            random_state=SEED,
            verbose=0,
        )
        model.fit(Xs, ys)
        kmm_models[name] = model
    except Exception as e:
        print(f"  Failed for {name}: {e}")

print(f"\nTrained {len(kmm_models)} KMM models.")

## 11. Per-source KMM evaluation on target test set

In [ ]:
kmm_per_source = []
for name, model in kmm_models.items():
    y_pred = np.asarray(model.predict(X_target_test_s)).astype(int).ravel()
    acc = accuracy_score(y_target_test, y_pred)
    f1 = f1_score(y_target_test, y_pred, zero_division=0)
    kmm_per_source.append({"source": name, "accuracy": acc, "f1": f1})

kmm_results_df = pd.DataFrame(kmm_per_source).sort_values("accuracy", ascending=False)
print(kmm_results_df.to_string(index=False))

## 12. KMM ensemble

In [ ]:
kmm_probs = np.stack([predict_proba_safe(m, X_target_test_s) for m in kmm_models.values()], axis=0)
kmm_ensemble_proba = kmm_probs.mean(axis=0)
kmm_ensemble_pred = (kmm_ensemble_proba >= 0.5).astype(int)

kmm_ens_acc = accuracy_score(y_target_test, kmm_ensemble_pred)
kmm_ens_f1 = f1_score(y_target_test, kmm_ensemble_pred, zero_division=0)
try:
    kmm_ens_auc = roc_auc_score(y_target_test, kmm_ensemble_proba)
except ValueError:
    kmm_ens_auc = float("nan")

print(f"KMM ensemble accuracy: {kmm_ens_acc:.3f}")
print(f"KMM ensemble F1:       {kmm_ens_f1:.3f}")
print(f"KMM ensemble AUC:      {kmm_ens_auc:.3f}")

## 13. Final comparison: baseline vs TrAdaBoost vs KMM

In [ ]:
comparison = pd.DataFrame([
    {"Method": "No-transfer baseline", "Accuracy": base_acc,    "F1": base_f1,    "AUC": base_auc},
    {"Method": "TrAdaBoost ensemble", "Accuracy": ens_acc,     "F1": ens_f1,     "AUC": ens_auc},
    {"Method": "KMM ensemble",        "Accuracy": kmm_ens_acc, "F1": kmm_ens_f1, "AUC": kmm_ens_auc},
])
print(comparison.to_string(index=False))

# Bar chart comparison
fig_cmp = go.Figure()
metrics = ["Accuracy", "F1", "AUC"]
for m in metrics:
    fig_cmp.add_trace(go.Bar(name=m, x=comparison["Method"], y=comparison[m]))
fig_cmp.update_layout(
    title="Method Comparison on Target Test Set",
    barmode="group",
    yaxis=dict(range=[0, 1]),
    width=900, height=500,
)
fig_cmp.show()

## 14. Per-source accuracy comparison (TrAdaBoost vs KMM)

In [ ]:
merged = results_df.rename(columns={"accuracy": "TrAdaBoost", "f1": "TrAdaBoost_f1"}).merge(
    kmm_results_df.rename(columns={"accuracy": "KMM", "f1": "KMM_f1"}),
    on="source", how="outer",
).sort_values("TrAdaBoost", ascending=False)

fig_src = go.Figure()
fig_src.add_trace(go.Bar(x=merged["source"], y=merged["TrAdaBoost"], name="TrAdaBoost", marker_color="steelblue"))
fig_src.add_trace(go.Bar(x=merged["source"], y=merged["KMM"], name="KMM", marker_color="orange"))
fig_src.add_hline(y=base_acc, line_dash="dash", line_color="red",
                  annotation_text=f"baseline={base_acc:.3f}", annotation_position="top right")
fig_src.update_layout(
    title="Per-Source Target-Test Accuracy: TrAdaBoost vs KMM",
    xaxis_title="Source task", yaxis_title="Accuracy",
    xaxis_tickangle=-45, height=550, width=1200,
    barmode="group", yaxis=dict(range=[0, 1]),
)
fig_src.show()

## 15. KMM ensemble decision boundary (focal_position fixed)

In [ ]:
kmm_probs_grid = np.stack([predict_proba_safe(m, grid_s) for m in kmm_models.values()], axis=0)
p_kmm_ensemble = kmm_probs_grid.mean(axis=0)

plot_decision_isosurface(
    p_kmm_ensemble,
    "KMM Ensemble — P(feasible) iso-surfaces"
).show()

## 16. Per-source Optimal Transport (SinkhornL1l2Transport)

For each source task, fit a class-regularized OT plan transporting source samples into the target feature space, then train a fresh decision-tree classifier on the transported source data.

`SinkhornL1l2Transport` uses entropic regularization (`reg_e`) plus a class-aware group-sparse penalty (`reg_cl`) so that source samples of different classes don't get mapped to the same target points.

In [ ]:
from ot.da import SinkhornL1l2Transport, JCPOTTransport

ot_models = {}        # one classifier per source
ot_transports = {}    # store the OT object so we can reuse for grid prediction

for name, (Xs, ys) in source_data.items():
    try:
        transport = SinkhornL1l2Transport(reg_e=1.0, reg_cl=0.1, max_iter=20)
        transport.fit(Xs=Xs, ys=ys, Xt=X_target_train_s)
        Xs_transported = transport.transform(Xs=Xs)

        clf = make_base_estimator()
        clf.fit(Xs_transported, ys)

        ot_models[name] = clf
        ot_transports[name] = transport
    except Exception as e:
        print(f"  Failed for {name}: {e}")

print(f"\nTrained {len(ot_models)} OT (SinkhornL1l2) models.")

## 17. Per-source OT evaluation on target test set

In [ ]:
ot_per_source = []
for name, clf in ot_models.items():
    y_pred = np.asarray(clf.predict(X_target_test_s)).astype(int).ravel()
    acc = accuracy_score(y_target_test, y_pred)
    f1 = f1_score(y_target_test, y_pred, zero_division=0)
    ot_per_source.append({"source": name, "accuracy": acc, "f1": f1})

ot_results_df = pd.DataFrame(ot_per_source).sort_values("accuracy", ascending=False)
print(ot_results_df.to_string(index=False))

## 18. OT ensemble (average probabilities)

In [ ]:
ot_probs = np.stack([predict_proba_safe(m, X_target_test_s) for m in ot_models.values()], axis=0)
ot_ensemble_proba = ot_probs.mean(axis=0)
ot_ensemble_pred = (ot_ensemble_proba >= 0.5).astype(int)

ot_ens_acc = accuracy_score(y_target_test, ot_ensemble_pred)
ot_ens_f1 = f1_score(y_target_test, ot_ensemble_pred, zero_division=0)
try:
    ot_ens_auc = roc_auc_score(y_target_test, ot_ensemble_proba)
except ValueError:
    ot_ens_auc = float("nan")

print(f"OT (SinkhornL1l2) ensemble accuracy: {ot_ens_acc:.3f}")
print(f"OT ensemble F1:                      {ot_ens_f1:.3f}")
print(f"OT ensemble AUC:                     {ot_ens_auc:.3f}")

## 19. Multi-source OT: JCPOTTransport

JCPOT (Redko et al. 2019) jointly transports **all** source distributions to the target in a single OT problem with class-proportion constraints. This is the multi-source extension of the JDOT idea built into POT.

In [ ]:
Xs_list = [Xs for (Xs, ys) in source_data.values()]
ys_list = [ys for (Xs, ys) in source_data.values()]

jcpot = JCPOTTransport(reg_e=0.1, max_iter=20, tol=1e-6)
jcpot.fit(Xs=Xs_list, ys=ys_list, Xt=X_target_train_s)

# Transport each source separately, then pool with labels
Xs_jcpot_transported = jcpot.transform(Xs=Xs_list)
X_jcpot_pooled = np.vstack(Xs_jcpot_transported)
y_jcpot_pooled = np.concatenate(ys_list)

print(f"JCPOT pooled training set: {X_jcpot_pooled.shape[0]} samples")

# Train final classifier on transported pooled sources + labeled target
X_jcpot_train = np.vstack([X_jcpot_pooled, X_target_train_s])
y_jcpot_train = np.concatenate([y_jcpot_pooled, y_target_train])

jcpot_clf = make_base_estimator()
jcpot_clf.fit(X_jcpot_train, y_jcpot_train)

jcpot_pred = jcpot_clf.predict(X_target_test_s)
jcpot_proba = jcpot_clf.predict_proba(X_target_test_s)[:, 1]
jcpot_acc = accuracy_score(y_target_test, jcpot_pred)
jcpot_f1 = f1_score(y_target_test, jcpot_pred, zero_division=0)
try:
    jcpot_auc = roc_auc_score(y_target_test, jcpot_proba)
except ValueError:
    jcpot_auc = float("nan")

print(f"JCPOT classifier accuracy: {jcpot_acc:.3f}")
print(f"JCPOT classifier F1:       {jcpot_f1:.3f}")
print(f"JCPOT classifier AUC:      {jcpot_auc:.3f}")

## 20. Final 4-way comparison: baseline vs TrAdaBoost vs KMM vs OT vs JCPOT

In [ ]:
comparison = pd.DataFrame([
    {"Method": "No-transfer baseline",   "Accuracy": base_acc,    "F1": base_f1,    "AUC": base_auc},
    {"Method": "TrAdaBoost ensemble",    "Accuracy": ens_acc,     "F1": ens_f1,     "AUC": ens_auc},
    {"Method": "KMM ensemble",           "Accuracy": kmm_ens_acc, "F1": kmm_ens_f1, "AUC": kmm_ens_auc},
    {"Method": "SinkhornL1l2 ensemble",  "Accuracy": ot_ens_acc,  "F1": ot_ens_f1,  "AUC": ot_ens_auc},
    {"Method": "JCPOT (multi-source)",   "Accuracy": jcpot_acc,   "F1": jcpot_f1,   "AUC": jcpot_auc},
])
print(comparison.to_string(index=False))

fig_cmp_all = go.Figure()
for m in ["Accuracy", "F1", "AUC"]:
    fig_cmp_all.add_trace(go.Bar(name=m, x=comparison["Method"], y=comparison[m]))
fig_cmp_all.update_layout(
    title="All Methods — Target Test Set Comparison",
    barmode="group",
    yaxis=dict(range=[0, 1]),
    width=1100, height=500,
)
fig_cmp_all.show()

## 21. Per-source comparison: TrAdaBoost vs KMM vs OT

In [ ]:
merged_all = (
    results_df.rename(columns={"accuracy": "TrAdaBoost"}).drop(columns=["f1"])
    .merge(kmm_results_df.rename(columns={"accuracy": "KMM"}).drop(columns=["f1"]),
           on="source", how="outer")
    .merge(ot_results_df.rename(columns={"accuracy": "SinkhornL1l2"}).drop(columns=["f1"]),
           on="source", how="outer")
    .sort_values("TrAdaBoost", ascending=False)
)

fig_3way = go.Figure()
fig_3way.add_trace(go.Bar(x=merged_all["source"], y=merged_all["TrAdaBoost"], name="TrAdaBoost", marker_color="steelblue"))
fig_3way.add_trace(go.Bar(x=merged_all["source"], y=merged_all["KMM"], name="KMM", marker_color="orange"))
fig_3way.add_trace(go.Bar(x=merged_all["source"], y=merged_all["SinkhornL1l2"], name="SinkhornL1l2", marker_color="purple"))
fig_3way.add_hline(y=base_acc, line_dash="dash", line_color="red",
                   annotation_text=f"baseline={base_acc:.3f}", annotation_position="top right")
fig_3way.update_layout(
    title="Per-Source Target-Test Accuracy: TrAdaBoost vs KMM vs Sinkhorn OT",
    xaxis_title="Source task", yaxis_title="Accuracy",
    xaxis_tickangle=-45, height=550, width=1300,
    barmode="group", yaxis=dict(range=[0, 1]),
)
fig_3way.show()

## 22. JCPOT decision boundary (focal_position fixed)

In [ ]:
p_jcpot_grid = jcpot_clf.predict_proba(grid_s)[:, 1]

plot_decision_isosurface(
    p_jcpot_grid,
    "JCPOT (multi-source OT) — P(feasible) iso-surfaces"
).show()